# Merging & Joining — merge(), join()

## Definition
`pd.merge()` combines DataFrames based on **matching key columns** — like SQL JOINs.

### Join Types
```
INNER JOIN: Only matching rows from both tables
LEFT JOIN:  All rows from left, matching from right (NaN if no match)
RIGHT JOIN: All rows from right, matching from left (NaN if no match)
OUTER JOIN: All rows from both (NaN where no match)
```

### merge() vs join()
| | `pd.merge()` | `df.join()` |
|--|------------|------------|
| Join on | Any column | Index by default |
| Flexibility | High | Simpler syntax |
| Typical use | Column-based joins | Index-based joins |

### Key Parameters
| Parameter | Description |
|-----------|-------------|
| `on` | Column(s) to join on (must exist in both) |
| `left_on` / `right_on` | Different key names in each table |
| `how` | 'inner', 'left', 'right', 'outer', 'cross' |
| `suffixes` | Suffixes for overlapping columns |
| `validate` | Check join type constraints |
| `indicator` | Add `_merge` column showing origin |

In [ ]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    'emp_id': [1, 2, 3, 4, 5],
    'name':   ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept_id': [10, 20, 10, 30, 20]
})

departments = pd.DataFrame({
    'dept_id':   [10, 20, 30, 40],
    'dept_name': ['Engineering', 'Marketing', 'HR', 'Finance'],
    'location':  ['NYC', 'LA', 'Chicago', 'Seattle']
})

print('Employees:'); print(employees)
print('\nDepartments:'); print(departments)

In [ ]:
# INNER JOIN — only matching rows
inner = pd.merge(employees, departments, on='dept_id', how='inner')
print('INNER JOIN:')
print(inner)

In [ ]:
# LEFT JOIN — all employees, NaN for unmatched depts
left = pd.merge(employees, departments, on='dept_id', how='left')
print('LEFT JOIN:')
print(left)

In [ ]:
# OUTER JOIN — all rows from both tables
outer = pd.merge(employees, departments, on='dept_id', how='outer')
print('OUTER JOIN:')
print(outer)

In [ ]:
# Merging on different column names
sales = pd.DataFrame({
    'order_id': [101, 102, 103],
    'customer_id': [1, 2, 1],
    'amount': [500, 300, 800]
})
customers = pd.DataFrame({
    'cid': [1, 2, 3],
    'customer_name': ['Alice', 'Bob', 'Carol']
})
merged = pd.merge(sales, customers, left_on='customer_id', right_on='cid', how='left')
print('Different key names:')
print(merged)

In [ ]:
# indicator=True — show which table row came from
merged_ind = pd.merge(employees, departments, on='dept_id', how='outer', indicator=True)
print('With indicator:')
print(merged_ind[['name', 'dept_name', '_merge']])

In [ ]:
# Merging on multiple keys
orders = pd.DataFrame({
    'year': [2022, 2022, 2023, 2023],
    'month': [1, 2, 1, 2],
    'revenue': [100, 150, 200, 175]
})
budget = pd.DataFrame({
    'year': [2022, 2022, 2023],
    'month': [1, 2, 1],
    'budget': [90, 140, 220]
})
merged_multi = pd.merge(orders, budget, on=['year', 'month'], how='left')
print('Multi-key merge:')
print(merged_multi)

In [ ]:
# validate — catch unexpected duplicates in join keys
try:
    pd.merge(employees, departments, on='dept_id', how='inner', validate='1:1')
except pd.errors.MergeError as e:
    print('MergeError (not 1:1):', str(e)[:80])

# 1:m means each dept_id in left appears once (it does for departments)
merged_validated = pd.merge(departments, employees, on='dept_id', how='inner', validate='1:m')
print('1:m validated merge:', merged_validated.shape)

In [ ]:
# df.join() — index-based join
df1 = pd.DataFrame({'A': [1, 2, 3]}, index=['x', 'y', 'z'])
df2 = pd.DataFrame({'B': [10, 20, 30]}, index=['y', 'z', 'w'])

print('join (left by default):')
print(df1.join(df2))   # joins on index

print('\njoin (outer):')
print(df1.join(df2, how='outer'))

In [ ]:
# EXCEPTION: ambiguous column names
df_a = pd.DataFrame({'key': [1, 2], 'value': [10, 20]})
df_b = pd.DataFrame({'key': [1, 2], 'value': [100, 200]})

# Without suffixes -> ValueError
try:
    pd.merge(df_a, df_b, on='key', suffixes=(None, None))
except Exception as e:
    print('Error:', type(e).__name__, str(e)[:60])

# With suffixes
result = pd.merge(df_a, df_b, on='key', suffixes=('_left', '_right'))
print('With suffixes:')
print(result)